In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
import json

In [ ]:
data = pd.read_excel('../../dataset/all data.xlsx',sheet_name='paperID_update')

In [20]:
data['logIpc'] = np.log10(data['Ipc'])
data['logSRC'] = np.log10(data['Soil residual concentration'])
data['logED'] = np.log10(data['ED'])

In [10]:
data.drop(columns=['Reference', 'Sample','Soil source','ES','PFAS','PFAS class','Species','Plant class','group','Ipc','Original tissue',
                   'ED','Soil residual concentration'], inplace=True)
X = data.drop(columns=['lnBCF'], errors='ignore').copy()
y = data['lnBCF'].copy()

In [11]:
cat_cols = [
    'Tissue',
    'Family','Genus','Order','monocot','woody','herb','crop','vegetable','legume','grass','perennial','edible'
]
cat_cols = [c for c in cat_cols if c in X.columns] 
cat_idx = [X.columns.get_loc(c) for c in cat_cols]

In [7]:
best_params = {
    'iterations': 1803,
    'depth': 7,
    'learning_rate': 0.03823231749308767,
    'l2_leaf_reg': 7.85887570845998,
    'bagging_temperature': 0.8606437364021795,
    'random_strength': 8.94247879276211,
    'subsample': 0.8029919474739348,
    'colsample_bylevel': 0.7100100060503998,

    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'verbose': 0,
    'random_seed': 42
}

In [10]:
final_model = CatBoostRegressor(**best_params)

final_model.fit(
    X,
    y,
    cat_features=cat_idx
)
final_model.save_model("../Model/catboost_china_prediction.cbm")

with open("../Model/features_china_prediction.json", "w") as f:
    json.dump(list(X.columns), f)

In [22]:
data_new = data.drop(
    columns=[
        'Reference',
        'Sample',
        'Soil source',
        'ES',
        'PFAS',
        'PFAS class',
        'Species',
        'Plant class',
        'group',
        'Ipc',
        'Original tissue',
        'ED',
        'Soil residual concentration',
        'logSRC'
    ],
    errors='ignore'
).copy()
X_new = data_new.drop(columns=['lnBCF'], errors='ignore').copy()
y_new = data_new['lnBCF'].copy()

In [24]:
cat_cols_new = [c for c in cat_cols if c in X_new.columns] 
cat_idx_new = [X_new.columns.get_loc(c) for c in cat_cols_new]

In [ ]:
final_model_no_src = CatBoostRegressor(**best_params)

final_model_no_src.fit(
    X_new,
    y_new,
    cat_features=cat_idx_new
)
final_model_no_src.save_model("../../Model/catboost_china_prediction_no_src.cbm")
    
with open("../../Model/features_china_prediction_no_src.json", "w") as f:
    json.dump(list(X_new.columns), f)